In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Data/result_for_model.csv")
df = df.drop(columns=['descendants', 'amount', 'amount_filled', 'month'], errors='ignore')

df.head()

,order_date,variety_id,grade,order_time,source_id,inventory,total_inventory,season
0,14010315,56,1,06:00-08:59,56,1.51,1.51,1
1,14010315,56,1,09:00-11:59,56,1.51,1.51,1
2,14010315,56,1,12:00-14:59,56,1.51,1.51,1
3,14010315,56,1,15:00-17:59,56,1.51,1.51,1
4,14010315,56,1,18:00-20:59,56,1.51,1.51,1


In [2]:
time_map = {
    '06:00-08:59': 1,
    '09:00-11:59': 2,
    '12:00-14:59': 3,
    '15:00-17:59': 4,
    '18:00-20:59': 5,
    '21:00-23:59': 6,
}

df['order_range'] = df['order_time'].map(time_map)
df = df.drop(columns=['order_time'])
df.head()

,order_date,variety_id,grade,source_id,inventory,total_inventory,season,order_range
0,14010315,56,1,56,1.51,1.51,1,1
1,14010315,56,1,56,1.51,1.51,1,2
2,14010315,56,1,56,1.51,1.51,1,3
3,14010315,56,1,56,1.51,1.51,1,4
4,14010315,56,1,56,1.51,1.51,1,5


In [3]:
X = df.drop(columns=['inventory'])
y = df['inventory']

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def train_and_evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "mae": mean_absolute_error(y_test, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "r2": r2_score(y_test, y_pred)
    }

    return metrics
    

In [6]:
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor

models = {
    "RandomForestRegressor": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-2),
    "LinearRegression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=42),
    #"ExtraTreesRegressor": ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    #"XGBRegressor": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1),
}

def run_model(name, model):
    metrics = train_and_evaluate(model, X_train, X_test, y_train, y_test)
    return name, metrics


results = {}

# for name, model in models.items():
#     results[name] = run_model(name, model)
    
results_list = Parallel(n_jobs=-2)(
    delayed(run_model)(name, model)
    for name, model in models.items()
)
results = dict(results_list)

In [7]:
results_df = pd.DataFrame(results).T
print(results_df)

                                    mae       rmse        r2
RandomForestRegressor          0.780418   4.745663  0.989784
LinearRegression               5.681072  27.696444  0.652034
DecisionTreeRegressor          0.681557   6.244182  0.982314
GradientBoostingRegressor      1.792987   8.395473  0.968027
HistGradientBoostingRegressor  1.785033  17.409972  0.862505


In [21]:
import joblib

rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=5, min_samples_split=10, max_features="sqrt", random_state=42, n_jobs=1)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

metrics = {
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
    "r2": r2_score(y_test, y_pred)
}

joblib.dump(rf_model, "random_forest.pkl")

metrics

{'mae': 1.7587748229668907,
 'rmse': np.float64(8.335330254291396),
 'r2': 0.9684837144991972}